# test with doi data set and existing plugins

In [1]:
from doi_downloader import loader as ld, doi_downloader as ddl, pdf_download as pdf_dl
import json
import os
import polars as pl
import regex
from termcolor import colored
import time
import urllib

In [2]:
plugins = ld.plugins
PLUGINS = {"CoreacukPlugin": plugins['CoreacukPlugin'],
           "CrossrefPlugin": plugins['CrossrefPlugin'],
           "DoiorgPlugin": plugins['DoiorgPlugin'],
           "GoogleScholarSerpAPIPlugin": plugins['GoogleScholarSerpAPIPlugin'],
           "UnpaywallPlugin": plugins['UnpaywallPlugin']} # requires a commercial api key

CHAR_SUCCESS = "✅"
CHAR_FAILURE = "❌"

In [3]:
def make_safe_filename(doi):
    """Replace slashes and points in DOI by underscores, Add .pdf"""
    return doi.replace("/", "_").replace(".", "_") + ".pdf"

In [4]:
def get_pdf_urls_from_all_plugins(doi, debug=False):
    """Retrieve pdf_urls with all plugins"""
    urls = {}
    for plugin_name in PLUGINS.keys():
        try:
            plugin_data = PLUGINS[plugin_name].get_pdf_url(doi, use_cache=(plugin_name != "DoiorgPlugin"))
            try:
                plugin_data_json = json.loads(plugin_data)
                urls[plugin_name] = plugin_data_json["pdf_links"][0]
            except:
                urls[plugin_name] = plugin_data
            if debug: print("DEBUG:", plugin_name, urls[plugin_name], (plugin_name != "DoiorgPlugin"), urls)
        except Exception as e:
            print(f"   get_pdf_urls: error: {e}")
    return urls

In [5]:
def summarize_plugin_results(pdf_urls, index, result_type="url"):
    """Summarize plugin results with single icon: CHAR_SUCCESS or CHAR_FAILURE"""
    prefix =  f"   " if result_type == "url" else "   "
    print(f"{prefix} {result_type} retrieval success per plugin:", end=" ")
    for plugin_name in sorted(pdf_urls):
        print((CHAR_SUCCESS if pdf_urls[plugin_name] else CHAR_FAILURE), end=" ")
    print()

In [6]:
def download_pdf(doi, pdf_target_dir_name, pdf_target_file_name, pdf_urls):
    """Try to download pdf with each plugin, Register result"""
    target_locations = {}
    for plugin_name in pdf_urls:
        if pdf_urls[plugin_name]:
            target_locations[plugin_name] = pdf_dl.download_pdf(pdf_urls[plugin_name], 
                                                                pdf_target_file_name, 
                                                                pdf_target_dir_name)
    for plugin_name in PLUGINS:
        if plugin_name not in target_locations or not target_locations[plugin_name]:
            target_locations[plugin_name] = None
    return target_locations

In [7]:
def check_plugin_access(doi_df):
    """Check plugin access to a list of DOIs: url retrieval and pdf downloads"""
    target_locations = []
    pdf_urls = []
    pdf_target_dir_name = "../downloads"
    for index, row in enumerate(doi_df.iter_rows()):
        doi = row[0]
        print(f"{index + 1}. {doi}")
        pdf_target_file_name = make_safe_filename(doi)
        pdf_urls.append(get_pdf_urls_from_all_plugins(doi, debug=False))
        summarize_plugin_results({key[0]: pdf_urls[-1][key[0]] 
                                  for key in sorted(pdf_urls[-1].items(), 
                                                    key=lambda item: item[0])}, 
                                 index + 1)
        target_locations.append(download_pdf(doi,
                                             pdf_target_dir_name, 
                                             pdf_target_file_name, 
                                             pdf_urls[-1]))
        summarize_plugin_results({key[0]: target_locations[-1][key[0]] 
                                          for key in sorted(target_locations[-1].items(), 
                                                     key=lambda item: item[0])}, 
                                          index + 1, result_type="pdf")
        if index >= MAX_PROCESSED:
             break
    return pdf_urls, target_locations

The test DOIs were taken from the file `doi_coda.csv`

In [8]:
doi_df = pl.DataFrame([{"doi": '10.1177/0022002714560349'},
                       {"doi": '10.1007/s10198-013-0496-x'},
                       {"doi": '10.1111/j.1465-7295.2010.00309.x'},
                       {"doi": '10.1016/j.econlet.2009.08.024'},
                       {"doi": '10.1038/s41467-017-00731-0'},
                       {"doi": '10.1093/ei/cbl001'},
                       {"doi": '10.1016/j.jesp.2004.09.004'},
                       {"doi": '10.1016/j.jpubeco.2014.01.005'},
                       {"doi": '10.1111/puar.12424'},
                       {"doi": '10.1016/j.geb.2006.09.005'},
                       {"doi": '10.1016/j.joep.2007.01.007'},
                       {"doi": '10.1108/03068290810854565'},
                       {"doi": '10.1111/j.1468-0335.2008.00689'},
                       {"doi": '10.1177/1069397108322455'},
                       {"doi": '10.1016/j.socec.2010.12.013'},
                       {"doi": '10.1017/s0022381609090355'},
                       {"doi": '10.1016/j.jpubeco.2008.06.007'},
                       {"doi": '10.1093/restud/rdt017'},
                       {"doi": '10.1016/j.socec.2011.08.028'},
                       {"doi": '10.1093/jae/ejr034'}])          

In [11]:
MAX_PROCESSED = 20

In [12]:
pdf_urls, target_locations = check_plugin_access(doi_df[:MAX_PROCESSED])

1. 10.1177/0022002714560349
[coreacuk] Unauthorized access. Check your API key.
[crossref] using cached data for 10.1177/0022002714560349.
[doi.org] An error occurred: 403 Client Error: Forbidden for url: https://journals.sagepub.com/doi/10.1177/0022002714560349
[serpapi] using cached data for 10.1177/0022002714560349.
[unpaywall] using cached data for 10.1177/0022002714560349.
    url retrieval success per plugin: ❌ ✅ ❌ ✅ ❌ 
    pdf retrieval success per plugin: ❌ ❌ ❌ ❌ ❌ 
2. 10.1007/s10198-013-0496-x
[coreacuk] Unauthorized access. Check your API key.
[crossref] using cached data for 10.1007/s10198-013-0496-x.
[doi.org] robots.txt blocked acccess to https://link.springer.com/article/10.1007/s10198-013-0496-x
[serpapi] using cached data for 10.1007/s10198-013-0496-x.
[unpaywall] using cached data for 10.1007/s10198-013-0496-x.
    url retrieval success per plugin: ❌ ✅ ❌ ✅ ❌ 
    pdf retrieval success per plugin: ❌ ❌ ❌ ❌ ❌ 
3. 10.1111/j.1465-7295.2010.00309.x
[coreacuk] Unauthorized ac

In [13]:
pl.Config.set_tbl_rows(100)
df_pdf_urls = pl.DataFrame(pdf_urls)
aggregated_pdf_urls = {col: len(df_pdf_urls) - val for col, val in df_pdf_urls.null_count().row(0, named=True).items()}
aggregated_pdf_urls["Any"] = len([True for row in df_pdf_urls.iter_rows() if set(row) != {None}])
pl.DataFrame(aggregated_pdf_urls)

CoreacukPlugin,CrossrefPlugin,DoiorgPlugin,GoogleScholarSerpAPIPlugin,UnpaywallPlugin,Any
i64,i64,i64,i64,i64,i64
0,5,1,20,4,20


In [14]:
df_target_locations = pl.DataFrame(target_locations)
df_target_locations = df_target_locations.select(sorted(df_target_locations.columns))
aggregated_target_locations = {col: len(df_target_locations) - 
                               val for col, val in df_target_locations.null_count().row(0, named=True).items()}
aggregated_target_locations["Any"] = len([True for row in df_target_locations.iter_rows() if set(row) != {None}])
pl.DataFrame(aggregated_target_locations)

CoreacukPlugin,CrossrefPlugin,DoiorgPlugin,GoogleScholarSerpAPIPlugin,UnpaywallPlugin,Any
i64,i64,i64,i64,i64,i64
0,1,1,4,3,5
